# TMJ Position Classifier — 2D Multi-View (v4)

Классификация положения головок ВНЧС по 2D-срезам из 3D-кропов.

**Задача:** для каждого сустава предсказать 2 метки — сагиттальное и фронтальное положение (3 класса каждое).

**Подход:** из каждого 128³ кропа берём 3 центральных среза (axial, coronal, sagittal),
пропускаем через pretrained ResNet18, используем features двумя способами:

- **Approach C:** Frozen features → SVM / Random Forest / Gradient Boosting
- **Approach A:** Multi-view fine-tuning ResNet18 + FC heads

**Преимущества vs 3D CNN (v1–v3):**
- ImageNet pretraining даёт сильные features без необходимости учить с нуля
- 2D CNN на порядки эффективнее по данным, чем 3D
- Клинически обосновано: врач оценивает положение головки по 2D-проекциям

**Требования:**
- GPU runtime (T4): Runtime → Change runtime type → T4 GPU
- Готовые `.npy` кропы на Google Drive в `tmj_data/tmj_crops/`
- Кропы создаются ноутбуком `train_position_classifier.ipynb` (секции 1–3)

## 1. Setup

In [ ]:
%pip install -q scikit-learn matplotlib seaborn tqdm gdown

In [ ]:
import os
from pathlib import Path

_NUM_CPU = max(1, os.cpu_count() or 8)
for _k in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ.setdefault(_k, str(_NUM_CPU))

# --- Detect environment ---
IN_COLAB = False
IN_DATASPHERE = False

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

if not IN_COLAB and Path("/home/jupyter").exists():
    IN_DATASPHERE = True

# --- Data root (read-only input data) ---
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/tmj_data")
elif IN_DATASPHERE:
    DATA_ROOT = Path("/home/jupyter/datasets/tmj_data")
else:
    DATA_ROOT = Path("./data/tmj_data")

# --- Writable directories for cache and outputs ---
if IN_COLAB:
    WORK_ROOT = Path("/content")
elif IN_DATASPHERE:
    WORK_ROOT = Path("/home/jupyter/project")
else:
    WORK_ROOT = Path(".")

OUTPUT_DIR = str(WORK_ROOT / "experiments")
FEATURES_CACHE_DIR = WORK_ROOT / "tmj_2d_features"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
FEATURES_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def save_output(path):
    """Download in Colab, just print path elsewhere."""
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print(f"Saved: {Path(path).resolve()}")

env_name = "Google Colab" if IN_COLAB else ("DataSphere" if IN_DATASPHERE else "Local")
print(f"Environment:    {env_name}")
print(f"CPU threads:    {_NUM_CPU} (OpenMP/BLAS via env)")
print(f"DATA_ROOT:      {DATA_ROOT}")
print(f"OUTPUT_DIR:     {OUTPUT_DIR}")
print(f"FEATURES_CACHE: {FEATURES_CACHE_DIR}")

In [ ]:
if not DATA_ROOT.exists() or not (DATA_ROOT / "tmj_crops").exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_ROOT}.\n"
        f"  Colab: mount Google Drive with tmj_data folder.\n"
        f"  DataSphere: create dataset 'tmj_data' (see README).\n"
        f"  Local: place data in ./data/tmj_data/"
    )

n = len(list((DATA_ROOT / "tmj_crops").glob("*.npy")))
print(f"Data OK: {DATA_ROOT}  ({n} crops)")

In [ ]:
import os
import json
import random
import logging
import datetime
from pathlib import Path
from typing import Dict, List, Tuple
from collections import Counter

import numpy as np
import torch

_nc = globals().get("_NUM_CPU", max(1, os.cpu_count() or 8))
torch.set_num_threads(_nc)
torch.set_num_interop_threads(min(4, _nc))
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as T
from tqdm.auto import tqdm

DATASET_ROOT = str(DATA_ROOT / "dataset_public")
LABELS_PATH = str(DATA_ROOT / "tmj_position_labels.json")
MANIFEST_PATH = str(DATA_ROOT / "dataset_public" / "manifest_private.json")
CROPS_DIR = DATA_ROOT / "tmj_crops"
FEATURES_CACHE = FEATURES_CACHE_DIR

assert CROPS_DIR.exists(), f"Crops not found: {CROPS_DIR}. Run preprocessing first (or init DataSphere dataset)."
assert Path(LABELS_PATH).exists(), f"Labels not found: {LABELS_PATH}"
assert Path(MANIFEST_PATH).exists(), f"Manifest not found: {MANIFEST_PATH}"

print(f"Crops dir: {CROPS_DIR}  ({len(list(CROPS_DIR.glob('*.npy')))} files)")
print(f"Features cache: {FEATURES_CACHE}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU — Approach A will be slow, Approach C works fine on CPU.")

print(f"Device: {device}")

## 2. Label Table

Тот же join + split, что в 3D-ноутбуке. `seed=42, split_ratio=0.8` для прямого сравнения.

In [ ]:
logger = logging.getLogger("tmj_2d")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


def map_sagittal(code: int) -> int:
    if code not in (1, 2, 3):
        raise ValueError(f"Invalid sagittal code: {code!r}")
    return code - 1


def map_frontal(code: int) -> int:
    if code not in (4, 5, 6):
        raise ValueError(f"Invalid frontal code: {code!r}")
    return code - 4


def build_index(manifest_path, labels_path, dataset_root):
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    with open(labels_path, "r", encoding="utf-8") as f:
        labels_data = json.load(f)

    label_by_name = {}
    for patient in labels_data["patients"]:
        label_by_name[patient["name_raw"].strip()] = patient["labels"]

    records, skipped = [], 0
    for study in manifest["studies"]:
        name = study["patient_name"].strip()
        if name not in label_by_name:
            skipped += 1
            continue
        lbl = label_by_name[name]
        records.append({
            "study_id": study["study_id"],
            "dicom_dir": os.path.join(dataset_root, study["study_id"]),
            "patient_name": name,
            "sag_right": map_sagittal(lbl["sagittal"]["right"]),
            "sag_left":  map_sagittal(lbl["sagittal"]["left"]),
            "fr_right":  map_frontal(lbl["frontal"]["right"]),
            "fr_left":   map_frontal(lbl["frontal"]["left"]),
        })
    logger.info(f"build_index: {len(records)} matched, {skipped} skipped")
    return records


def split_by_patient(records, split_ratio=0.8, seed=42):
    patients = sorted(set(r["patient_name"] for r in records))
    n = len(patients)
    if n <= 1:
        return records, []
    rng = random.Random(seed)
    rng.shuffle(patients)
    split_idx = min(max(1, int(n * split_ratio)), n - 1)
    train_patients = set(patients[:split_idx])
    train = [r for r in records if r["patient_name"] in train_patients]
    val = [r for r in records if r["patient_name"] not in train_patients]
    logger.info(f"Split: train={len(train)} ({len(train_patients)} pat) / val={len(val)} ({n - len(train_patients)} pat)")
    return train, val


def expand_to_crop_records(study_records, crops_dir):
    out = []
    for r in study_records:
        for side in ("left", "right"):
            crop_path = crops_dir / f"{r['study_id']}_{side}.npy"
            if not crop_path.exists():
                continue
            out.append({
                "study_id": r["study_id"],
                "patient_name": r["patient_name"],
                "side": side,
                "crop_path": str(crop_path),
                "sag": r[f"sag_{side}"],
                "fr":  r[f"fr_{side}"],
            })
    return out

In [ ]:
all_records = build_index(MANIFEST_PATH, LABELS_PATH, DATASET_ROOT)
train_records, val_records = split_by_patient(all_records, split_ratio=0.8)

train_crops = expand_to_crop_records(train_records, CROPS_DIR)
val_crops = expand_to_crop_records(val_records, CROPS_DIR)

print(f"Studies:  {len(all_records)} total, {len(train_records)} train, {len(val_records)} val")
print(f"Crops:    {len(train_crops)} train, {len(val_crops)} val")

sag_counts = Counter(r["sag"] for r in train_crops)
fr_counts = Counter(r["fr"] for r in train_crops)
print(f"\nSagittal class counts (train): {dict(sorted(sag_counts.items()))}")
print(f"Frontal  class counts (train): {dict(sorted(fr_counts.items()))}")

HEAD_NAMES = ["sagittal", "frontal"]
CLASS_NAMES = ["Anterior/Medial", "Normal", "Posterior/Lateral"]
CLASS_LABELS = [0, 1, 2]
NUM_CLASSES = 3

## 3. Slice Extraction + Feature Extraction

Из каждого 128³ `.npy` кропа берём 3 центральных среза:
- **Axial** (Z = mid) — горизонтальный
- **Coronal** (Y = mid) — фронтальный
- **Sagittal** (X = mid) — сагиттальный

Каждый срез → resize 224×224 → 3 канала → ImageNet нормализация → ResNet18 → 512-dim features.

Итого на один кроп: **3 × 512 = 1536** features. Кэшируем на Drive.

In [ ]:
import torch.nn.functional as F

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def extract_views(crop_path: str) -> np.ndarray:
    """Load 128^3 crop, return 3 central slices as (3, 128, 128) float32."""
    vol = np.load(crop_path).astype(np.float32)
    mid = vol.shape[0] // 2
    axial    = vol[mid, :, :]       # Z = mid
    coronal  = vol[:, mid, :]       # Y = mid
    sagittal = vol[:, :, mid]       # X = mid
    return np.stack([axial, coronal, sagittal], axis=0)  # (3, 128, 128)


def prepare_slice_tensor(slice_2d: np.ndarray) -> torch.Tensor:
    """Single 128x128 slice → (3, 224, 224) normalized tensor."""
    t = torch.from_numpy(slice_2d).float().unsqueeze(0)  # (1, 128, 128)
    t = F.interpolate(t.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)[0]  # (1, 224, 224)
    t = t.expand(3, -1, -1)  # (3, 224, 224)
    t = (t - IMAGENET_MEAN) / IMAGENET_STD
    return t


def build_feature_extractor():
    """ResNet18 without final FC → outputs 512-dim features."""
    resnet = models.resnet18(weights="IMAGENET1K_V1")
    backbone = nn.Sequential(*list(resnet.children())[:-1])  # → (B, 512, 1, 1)
    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad = False
    return backbone


@torch.no_grad()
def extract_all_features(records, backbone, device, crop_batch_size=16):
    """Extract (N, 1536) feature matrix: 3 views × 512 dim per crop.

    Batches several crops per forward pass (better CPU/GPU utilization).
    """
    backbone = backbone.to(device)
    all_features = []
    n = len(records)

    for start in tqdm(range(0, n, crop_batch_size), desc="Extracting features"):
        chunk = records[start : start + crop_batch_size]
        tensors = []
        for rec in chunk:
            views = extract_views(rec["crop_path"])
            for i in range(3):
                tensors.append(prepare_slice_tensor(views[i]))
        batch = torch.stack(tensors).to(device)
        feats = backbone(batch).flatten(1).cpu().numpy()
        for j in range(len(chunk)):
            all_features.append(feats[3 * j : 3 * j + 3].reshape(-1))

    return np.array(all_features, dtype=np.float32)

In [ ]:
backbone = build_feature_extractor()
print(f"ResNet18 backbone loaded ({sum(p.numel() for p in backbone.parameters()):,} params, all frozen)")

train_feat_path = FEATURES_CACHE / "train_features.npy"
val_feat_path   = FEATURES_CACHE / "val_features.npy"

if train_feat_path.exists() and val_feat_path.exists():
    X_train = np.load(train_feat_path)
    X_val   = np.load(val_feat_path)
    print(f"Loaded cached features: train {X_train.shape}, val {X_val.shape}")
else:
    print("Extracting features (first run)...")
    X_train = extract_all_features(train_crops, backbone, device)
    X_val   = extract_all_features(val_crops,   backbone, device)
    np.save(train_feat_path, X_train)
    np.save(val_feat_path,   X_val)
    print(f"Saved features: train {X_train.shape}, val {X_val.shape}")

y_train_sag = np.array([r["sag"] for r in train_crops])
y_train_fr  = np.array([r["fr"]  for r in train_crops])
y_val_sag   = np.array([r["sag"] for r in val_crops])
y_val_fr    = np.array([r["fr"]  for r in val_crops])

print(f"\nTrain: {X_train.shape[0]} samples × {X_train.shape[1]} features")
print(f"Val:   {X_val.shape[0]} samples × {X_val.shape[1]} features")

In [ ]:
import matplotlib.pyplot as plt

sample_views = extract_views(train_crops[0]["crop_path"])
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ["Axial (Z=mid)", "Coronal (Y=mid)", "Sagittal (X=mid)"]
for ax, img, title in zip(axes, sample_views, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
rec0 = train_crops[0]
fig.suptitle(f"{rec0['side']} TMJ — sag={rec0['sag']}, fr={rec0['fr']}")
plt.tight_layout()
plt.show()

### Что именно «кормим» нейросети

Здесь **нет сегментационных масок** в классическом смысле: детектор уже вырезал куб **128³** вокруг головки сустава. В ResNet идут **нормализованные по интенсивности срезы КТ** (после ресайза 224×224 и нормализации ImageNet — как в коде `prepare_slice_tensor`).

- **Approach C / ранний этап:** 3 **центральных** среза (axial / coronal / sagittal).
- **Секция 4b (v5):** по **5 позициям** вдоль каждой оси (multi-slice).

Ниже — сырые срезы из `.npy` и сравнение «128 → как видит ResNet».

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

VIZ_SLICE_POS = [0.2, 0.35, 0.5, 0.65, 0.8]


def multislice_planes(crop_path: str, positions=VIZ_SLICE_POS):
    vol = np.load(crop_path).astype(np.float32)
    D = vol.shape[0]
    idx = [max(0, min(D - 1, int(p * D))) for p in positions]
    axials = np.stack([vol[i, :, :] for i in idx])
    coronals = np.stack([vol[:, i, :] for i in idx])
    sagittals = np.stack([vol[:, :, i] for i in idx])
    return axials, coronals, sagittals, idx


def tensor_imagenet_to_rgb(t: torch.Tensor) -> np.ndarray:
    mean = IMAGENET_MEAN.cpu()
    std = IMAGENET_STD.cpu()
    x = t.detach().cpu() * std + mean
    x = x.clamp(0, 1).numpy()
    return np.transpose(x, (1, 2, 0))


rng = np.random.default_rng(42)
sample_idx = rng.choice(len(train_crops), size=min(6, len(train_crops)), replace=False)

fig, axes = plt.subplots(len(sample_idx), 3, figsize=(9, 2.8 * len(sample_idx)))
if len(sample_idx) == 1:
    axes = axes.reshape(1, -1)
titles = ["Axial (mid)", "Coronal (mid)", "Sagittal (mid)"]
for row, si in enumerate(sample_idx):
    rec = train_crops[int(si)]
    views = extract_views(rec["crop_path"])
    for j in range(3):
        ax = axes[row, j]
        ax.imshow(views[j], cmap="gray")
        ax.set_title(titles[j] if row == 0 else "")
        ax.axis("off")
    axes[row, 0].set_ylabel(
        f"{rec['patient_name'][:12]}…\n{rec['side']}  sag={rec['sag']} fr={rec['fr']}",
        fontsize=8,
    )
fig.suptitle("Вход ResNet (3 центральных среза) — несколько кропов из train", fontsize=11)
plt.tight_layout()
plt.show()

rec = train_crops[0]
aa, cc, ss, idx = multislice_planes(rec["crop_path"])
fig, axes = plt.subplots(3, len(VIZ_SLICE_POS), figsize=(14, 6))
for col, k in enumerate(idx):
    axes[0, col].imshow(aa[col], cmap="gray")
    axes[0, col].set_title(f"axial i={k}")
    axes[1, col].imshow(cc[col], cmap="gray")
    axes[1, col].set_title(f"coronal i={k}")
    axes[2, col].imshow(ss[col], cmap="gray")
    axes[2, col].set_title(f"sagittal i={k}")
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle(
    f"Multi-slice (v5): позиции {VIZ_SLICE_POS} — {rec['side']} sag={rec['sag']} fr={rec['fr']}",
    fontsize=11,
)
plt.tight_layout()
plt.show()

sl = extract_views(rec["crop_path"])[0]
t224 = prepare_slice_tensor(sl)
rgb = tensor_imagenet_to_rgb(t224)
vol = np.load(rec["crop_path"]).astype(np.float32)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(sl, cmap="gray")
ax[0].set_title(f"Сырой срез 128×128 (axial mid)\n[{sl.min():.3f}, {sl.max():.3f}]")
ax[0].axis("off")
ax[1].imshow(rgb)
ax[1].set_title("После prepare_slice_tensor\n(224×224, ImageNet → RGB для просмотра)")
ax[1].axis("off")
ax[2].hist(vol.ravel(), bins=80, color="steelblue", alpha=0.85)
ax[2].set_title("Гистограмма интенсивностей\nво всём кропе 128³")
ax[2].set_xlabel("Значение в .npy")
ax[2].set_ylabel("Частота")
plt.tight_layout()
plt.show()

## 4. Approach C: Frozen Features + Classical ML

1536-dim feature vector → StandardScaler → SVM / RandomForest / GradientBoosting.

Отдельная модель на каждую голову (sagittal, frontal).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

classifiers = {
    "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced", C=1.0, gamma="scale"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", max_depth=10, random_state=42,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42,
    ),
}

results_c = {}

for clf_name, clf_template in classifiers.items():
    print(f"\n{'=' * 60}")
    print(f"{clf_name}")
    print(f"{'=' * 60}")

    head_results = {}
    for head, y_tr, y_vl in [("sagittal", y_train_sag, y_val_sag),
                              ("frontal",  y_train_fr,  y_val_fr)]:
        from sklearn.base import clone
        clf = clone(clf_template)
        clf.fit(X_train_sc, y_tr)

        preds = clf.predict(X_val_sc)
        acc = accuracy_score(y_vl, preds)
        report = classification_report(
            y_vl, preds, labels=CLASS_LABELS, target_names=CLASS_NAMES,
            zero_division=0, output_dict=True,
        )
        cm = confusion_matrix(y_vl, preds, labels=CLASS_LABELS)

        head_results[head] = {
            "accuracy": acc, "predictions": preds.tolist(),
            "report": report, "confusion_matrix": cm.tolist(),
            "classifier": clf,
        }

        print(f"\n  {head.upper()} accuracy: {acc:.3f}")
        print(classification_report(
            y_vl, preds, labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0,
        ))

    mean_acc = (head_results["sagittal"]["accuracy"] + head_results["frontal"]["accuracy"]) / 2
    head_results["mean_accuracy"] = mean_acc
    results_c[clf_name] = head_results
    print(f"  MEAN accuracy: {mean_acc:.3f}")

In [ ]:
best_clf_name = max(results_c, key=lambda k: results_c[k]["mean_accuracy"])
best_c = results_c[best_clf_name]
print(f"Best classical ML: {best_clf_name}  (mean_acc={best_c['mean_accuracy']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, head in zip(axes, HEAD_NAMES):
    cm = np.array(best_c[head]["confusion_matrix"])
    present = sorted(set(range(NUM_CLASSES)))
    names = [CLASS_NAMES[i] for i in present]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=names, yticklabels=names)
    acc = best_c[head]["accuracy"]
    ax.set(title=f"{head.title()} — {best_clf_name} (acc={acc:.3f})",
           xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.show()

print("\nAll classifiers summary:")
for name, res in sorted(results_c.items(), key=lambda x: -x[1]["mean_accuracy"]):
    s = res["sagittal"]["accuracy"]
    f = res["frontal"]["accuracy"]
    print(f"  {name:25s}  sag={s:.3f}  fr={f:.3f}  mean={res['mean_accuracy']:.3f}")

## 4b. Optimized Binary Classifier (v5)

Переход на бинарную классификацию + multi-slice features + patient-level CV + SVM tuning.

- **Sagittal:** class 0 (Anterior) vs class 1+2 (Normal + Posterior) → "Anterior" vs "Non-anterior"
- **Frontal:** class 0+1 (Anterior + Normal) vs class 2 (Posterior/Lateral) → "Non-lateral" vs "Lateral"

In [ ]:
BINARY_NAMES_SAG = ["Anterior", "Non-anterior"]
BINARY_NAMES_FR  = ["Non-lateral", "Lateral"]

all_crops = expand_to_crop_records(all_records, CROPS_DIR)

def to_binary_sag(label_3):
    """Sagittal: 0 stays 0 (Anterior), 1+2 become 1 (Non-anterior)."""
    return 0 if label_3 == 0 else 1

def to_binary_fr(label_3):
    """Frontal: 0+1 become 0 (Non-lateral), 2 stays 1 (Lateral)."""
    return 0 if label_3 in (0, 1) else 1

y_all_sag_bin = np.array([to_binary_sag(r["sag"]) for r in all_crops])
y_all_fr_bin  = np.array([to_binary_fr(r["fr"])   for r in all_crops])
groups_all    = np.array([r["patient_name"] for r in all_crops])

print("Binary label distribution (all data):")
print(f"  Sagittal: {dict(Counter(y_all_sag_bin))}  ({BINARY_NAMES_SAG})")
print(f"  Frontal:  {dict(Counter(y_all_fr_bin))}  ({BINARY_NAMES_FR})")
print(f"  Total crops: {len(all_crops)}, unique patients: {len(set(groups_all))}")

In [ ]:
SLICE_POSITIONS = [0.2, 0.35, 0.5, 0.65, 0.8]

def extract_views_multislice(crop_path, positions=SLICE_POSITIONS):
    """Load 128^3 crop, return (n_views=3, n_positions, 128, 128)."""
    vol = np.load(crop_path).astype(np.float32)
    D = vol.shape[0]
    indices = [max(0, min(D - 1, int(p * D))) for p in positions]
    axials    = np.stack([vol[i, :, :] for i in indices])     # (5, 128, 128)
    coronals  = np.stack([vol[:, i, :] for i in indices])     # (5, 128, 128)
    sagittals = np.stack([vol[:, :, i] for i in indices])     # (5, 128, 128)
    return np.stack([axials, coronals, sagittals])             # (3, 5, 128, 128)


@torch.no_grad()
def extract_multislice_features(
    records, backbone, device, positions=SLICE_POSITIONS, crop_batch_size=8
):
    """Extract features: mean-pooled (N, 1536) and concat (N, 7680).

    Batches multiple crops per forward (15 slices per crop × batch).
    """
    backbone = backbone.to(device)
    feats_mean, feats_concat = [], []
    n_pos = len(positions)
    n_sl = 3 * n_pos

    for start in tqdm(range(0, len(records), crop_batch_size), desc="Multi-slice features"):
        chunk = records[start : start + crop_batch_size]
        tensors = []
        for rec in chunk:
            views = extract_views_multislice(rec["crop_path"], positions)
            for v in range(3):
                for s in range(n_pos):
                    tensors.append(prepare_slice_tensor(views[v, s]))
        batch = torch.stack(tensors).to(device)
        out = backbone(batch).flatten(1).cpu().numpy()

        for j in range(len(chunk)):
            block = out[n_sl * j : n_sl * (j + 1)].reshape(3, n_pos, 512)
            mean_f = np.concatenate([block[v].mean(axis=0) for v in range(3)])
            concat_f = block.reshape(-1)
            feats_mean.append(mean_f)
            feats_concat.append(concat_f)

    return (
        np.array(feats_mean, dtype=np.float32),
        np.array(feats_concat, dtype=np.float32),
    )

In [ ]:
ms_mean_path   = FEATURES_CACHE / "all_multislice_mean.npy"
ms_concat_path = FEATURES_CACHE / "all_multislice_concat.npy"

if ms_mean_path.exists() and ms_concat_path.exists():
    X_all_mean   = np.load(ms_mean_path)
    X_all_concat = np.load(ms_concat_path)
    print(f"Loaded cached multi-slice features: mean {X_all_mean.shape}, concat {X_all_concat.shape}")
else:
    print("Extracting multi-slice features for ALL crops (first run)...")
    X_all_mean, X_all_concat = extract_multislice_features(all_crops, backbone, device)
    np.save(ms_mean_path, X_all_mean)
    np.save(ms_concat_path, X_all_concat)
    print(f"Saved: mean {X_all_mean.shape}, concat {X_all_concat.shape}")

print(f"\nFeature variants:")
print(f"  Mean-pooled: {X_all_mean.shape[1]} features (3 views × 512, avg over 5 slices)")
print(f"  Concat:      {X_all_concat.shape[1]} features (3 views × 5 slices × 512)")

In [ ]:
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

param_grid_base = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": ["scale", "auto", 0.001, 0.01],
    "svm__kernel": ["rbf", "linear"],
}

def run_cv_with_tuning(X, y, groups, feat_name, head_name, class_names):
    """Patient-level 5-fold CV with GridSearch inside each fold."""
    oof_preds = np.full(len(y), -1, dtype=int)
    fold_accs = []
    best_params_per_fold = []

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_vl = X[train_idx], X[val_idx]
        y_tr, y_vl = y[train_idx], y[val_idx]
        g_tr = groups[train_idx]

        inner_cv = GroupKFold(n_splits=3)
        inner_splits = list(inner_cv.split(X_tr, y_tr, g_tr))
        min_inner_train = min(len(tr) for tr, _ in inner_splits)
        safe_pca = [c for c in [20, 50, 80] if c < min_inner_train]
        if not safe_pca:
            safe_pca = [min(15, min_inner_train - 1)]

        param_grid = {**param_grid_base, "pca__n_components": safe_pca}

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("svm", SVC(class_weight="balanced", probability=True)),
        ])

        grid = GridSearchCV(
            pipe, param_grid, cv=inner_splits,
            scoring="accuracy", n_jobs=-1, refit=True,
        )
        grid.fit(X_tr, y_tr)

        preds = grid.predict(X_vl)
        oof_preds[val_idx] = preds
        fold_acc = accuracy_score(y_vl, preds)
        fold_accs.append(fold_acc)
        best_params_per_fold.append(grid.best_params_)

        print(f"  Fold {fold_i+1}: acc={fold_acc:.3f}  best_C={grid.best_params_['svm__C']}  "
              f"PCA={grid.best_params_['pca__n_components']}  "
              f"kernel={grid.best_params_['svm__kernel']}")

    mean_acc = np.mean(fold_accs)
    std_acc  = np.std(fold_accs)
    print(f"\n  {head_name} [{feat_name}]: {mean_acc:.3f} +- {std_acc:.3f}")

    return {
        "oof_preds": oof_preds,
        "fold_accs": fold_accs,
        "mean_acc": mean_acc,
        "std_acc": std_acc,
        "best_params": best_params_per_fold,
    }

In [ ]:
cv_results = {}

for feat_name, X_all in [("mean_pooled", X_all_mean), ("concat", X_all_concat)]:
    print(f"\n{'='*60}")
    print(f"Feature set: {feat_name} ({X_all.shape[1]} dims)")
    print(f"{'='*60}")

    for head_name, y_all, class_names in [
        ("Sagittal", y_all_sag_bin, BINARY_NAMES_SAG),
        ("Frontal",  y_all_fr_bin,  BINARY_NAMES_FR),
    ]:
        key = f"{feat_name}_{head_name.lower()}"
        print(f"\n--- {head_name} ---")
        cv_results[key] = run_cv_with_tuning(
            X_all, y_all, groups_all, feat_name, head_name, class_names
        )

print("\n" + "="*60)
print("SVM CV Summary")
print("="*60)
for key, res in cv_results.items():
    print(f"  {key:30s}: {res['mean_acc']:.3f} +- {res['std_acc']:.3f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

best_feat = max(
    ["mean_pooled", "concat"],
    key=lambda f: np.mean([
        cv_results[f"{f}_sagittal"]["mean_acc"],
        cv_results[f"{f}_frontal"]["mean_acc"],
    ])
)
X_best = X_all_mean if best_feat == "mean_pooled" else X_all_concat
print(f"Best feature set for ensemble: {best_feat} ({X_best.shape[1]} dims)")

def pick_best_svm_params(cv_res):
    """Select most common best params across folds."""
    from collections import Counter
    param_strs = [str(sorted(p.items())) for p in cv_res["best_params"]]
    most_common = Counter(param_strs).most_common(1)[0][0]
    return dict(eval(most_common))

ensemble_cv = {}

for head_name, y_all, class_names in [
    ("Sagittal", y_all_sag_bin, BINARY_NAMES_SAG),
    ("Frontal",  y_all_fr_bin,  BINARY_NAMES_FR),
]:
    svm_key = f"{best_feat}_{head_name.lower()}"
    bp = pick_best_svm_params(cv_results[svm_key])

    oof_preds = np.full(len(y_all), -1, dtype=int)
    fold_accs = []

    print(f"\n{'='*60}")
    print(f"Ensemble CV — {head_name}")
    print(f"{'='*60}")

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(X_best, y_all, groups_all)):
        X_tr, X_vl = X_best[train_idx], X_best[val_idx]
        y_tr, y_vl = y_all[train_idx], y_all[val_idx]

        scaler = StandardScaler().fit(X_tr)
        X_tr_s = scaler.transform(X_tr)
        X_vl_s = scaler.transform(X_vl)

        n_comp = min(bp.get("pca__n_components", 100), X_tr_s.shape[0], X_tr_s.shape[1])
        pca = PCA(n_components=n_comp).fit(X_tr_s)
        X_tr_p = pca.transform(X_tr_s)
        X_vl_p = pca.transform(X_vl_s)

        svm_clf = SVC(
            C=bp["svm__C"], gamma=bp["svm__gamma"], kernel=bp["svm__kernel"],
            class_weight="balanced", probability=True,
        )
        rf_clf = RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight="balanced",
            random_state=42,
        )
        gb_clf = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1,
            random_state=42,
        )
        ens = VotingClassifier(
            estimators=[("svm", svm_clf), ("rf", rf_clf), ("gb", gb_clf)],
            voting="soft",
        )
        ens.fit(X_tr_p, y_tr)
        preds = ens.predict(X_vl_p)
        oof_preds[val_idx] = preds
        fold_acc = accuracy_score(y_vl, preds)
        fold_accs.append(fold_acc)
        print(f"  Fold {fold_i+1}: acc={fold_acc:.3f}")

    mean_acc = np.mean(fold_accs)
    std_acc  = np.std(fold_accs)
    print(f"\n  Ensemble {head_name}: {mean_acc:.3f} +- {std_acc:.3f}")

    ensemble_cv[head_name.lower()] = {
        "oof_preds": oof_preds,
        "fold_accs": fold_accs,
        "mean_acc": mean_acc,
        "std_acc": std_acc,
    }

ens_mean = np.mean([ensemble_cv["sagittal"]["mean_acc"], ensemble_cv["frontal"]["mean_acc"]])
print(f"\n{'='*60}")
print(f"Ensemble mean accuracy: {ens_mean:.3f}")
print(f"{'='*60}")

In [ ]:
import seaborn as sns

# --- Pick the single best SVM config (feat+head combo) ---
best_svm_sag_key = max(
    [k for k in cv_results if k.endswith("_sagittal")],
    key=lambda k: cv_results[k]["mean_acc"],
)
best_svm_fr_key = max(
    [k for k in cv_results if k.endswith("_frontal")],
    key=lambda k: cv_results[k]["mean_acc"],
)
svm_sag = cv_results[best_svm_sag_key]
svm_fr  = cv_results[best_svm_fr_key]
ens_sag = ensemble_cv["sagittal"]
ens_fr  = ensemble_cv["frontal"]

# --- Confusion matrices (out-of-fold, all data) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for col, (method, sag_res, fr_res) in enumerate([
    ("SVM (tuned)", svm_sag, svm_fr),
    ("Ensemble",    ens_sag, ens_fr),
]):
    for row, (head, y_true, res, names) in enumerate([
        ("Sagittal", y_all_sag_bin, sag_res, BINARY_NAMES_SAG),
        ("Frontal",  y_all_fr_bin,  fr_res,  BINARY_NAMES_FR),
    ]):
        cm = confusion_matrix(y_true, res["oof_preds"], labels=[0, 1])
        ax = axes[row, col]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=names, yticklabels=names, ax=ax)
        acc = accuracy_score(y_true, res["oof_preds"])
        ax.set_title(f"{method} — {head}\nacc={acc:.3f} (5-fold OOF)")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

plt.tight_layout()
plt.savefig("v5_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Classification reports ---
for method, sag_res, fr_res in [
    ("SVM (tuned)", svm_sag, svm_fr),
    ("Ensemble",    ens_sag, ens_fr),
]:
    print(f"\n{'='*60}")
    print(f"{method} — Classification Reports (5-fold OOF)")
    print(f"{'='*60}")
    print(f"\nSagittal ({best_svm_sag_key}):")
    print(classification_report(y_all_sag_bin, sag_res["oof_preds"],
                                target_names=BINARY_NAMES_SAG, labels=[0, 1], zero_division=0))
    print(f"Frontal ({best_svm_fr_key}):")
    print(classification_report(y_all_fr_bin, fr_res["oof_preds"],
                                target_names=BINARY_NAMES_FR, labels=[0, 1], zero_division=0))

# --- Comparison with previous versions ---
print(f"\n{'='*60}")
print("Comparison with previous approaches")
print(f"{'='*60}")

svm_mean = np.mean([svm_sag["mean_acc"], svm_fr["mean_acc"]])

comparison_table = [
    ("v1 — 3D CNN baseline (3-class)",           "0.734", "32-sample val"),
    ("v3 — 3D CNN balanced (3-class)",            "0.734", "32-sample val"),
    ("v4 — Approach C: SVM+RF (3-class)",         "0.688", "32-sample val"),
    ("v4 — Approach A: Fine-tuned ResNet (3-cl)", "0.688", "32-sample val"),
    (f"v5 — SVM tuned (binary)",                  f"{svm_mean:.3f}", "5-fold CV (172 samples)"),
    (f"v5 — Ensemble (binary)",                   f"{ens_mean:.3f}", "5-fold CV (172 samples)"),
]

print(f"\n{'Approach':<45} {'Mean Acc':>8}  {'Eval Method'}")
print("-" * 80)
for name, acc, method in comparison_table:
    print(f"{name:<45} {acc:>8}  {method}")

In [ ]:
import pickle, datetime

svm_mean = np.mean([svm_sag["mean_acc"], svm_fr["mean_acc"]])
use_ensemble = ens_mean > svm_mean
final_method = "Ensemble" if use_ensemble else "SVM"
print(f"Training final model: {final_method} (mean acc = {max(ens_mean, svm_mean):.3f})")

scaler_final = StandardScaler().fit(X_best)
X_all_scaled = scaler_final.transform(X_best)

bp = pick_best_svm_params(cv_results[best_svm_sag_key])
n_comp = min(bp.get("pca__n_components", 100), X_all_scaled.shape[0], X_all_scaled.shape[1])
pca_final = PCA(n_components=n_comp).fit(X_all_scaled)
X_all_pca = pca_final.transform(X_all_scaled)

final_models = {}

for head_name, y_all, class_names in [
    ("sagittal", y_all_sag_bin, BINARY_NAMES_SAG),
    ("frontal",  y_all_fr_bin,  BINARY_NAMES_FR),
]:
    if use_ensemble:
        svm_clf = SVC(
            C=bp["svm__C"], gamma=bp["svm__gamma"], kernel=bp["svm__kernel"],
            class_weight="balanced", probability=True,
        )
        rf_clf = RandomForestClassifier(
            n_estimators=200, max_depth=10, class_weight="balanced", random_state=42,
        )
        gb_clf = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42,
        )
        clf = VotingClassifier(
            estimators=[("svm", svm_clf), ("rf", rf_clf), ("gb", gb_clf)],
            voting="soft",
        )
    else:
        clf = SVC(
            C=bp["svm__C"], gamma=bp["svm__gamma"], kernel=bp["svm__kernel"],
            class_weight="balanced", probability=True,
        )

    clf.fit(X_all_pca, y_all)
    train_acc = accuracy_score(y_all, clf.predict(X_all_pca))
    print(f"  {head_name}: train_acc={train_acc:.3f}")
    final_models[head_name] = clf

pipeline_dict = {
    "scaler": scaler_final,
    "pca": pca_final,
    "models": final_models,
    "feature_set": best_feat,
    "method": final_method,
    "class_names": {"sagittal": BINARY_NAMES_SAG, "frontal": BINARY_NAMES_FR},
    "slice_positions": SLICE_POSITIONS,
    "pca_n_components": n_comp,
    "svm_params": bp,
    "timestamp": datetime.datetime.now().isoformat(),
}

pipeline_path = OUTPUT_DIR + f"/position_classifier_v5_binary.pkl"
with open(pipeline_path, "wb") as f:
    pickle.dump(pipeline_dict, f)

print(f"\nSaved pipeline → {pipeline_path}")
print(f"Pipeline contents: scaler, PCA({n_comp}), {final_method} (sag+fr)")

save_output(pipeline_path)

In [ ]:
import json

v5_analysis = {
    "version": "v5_binary",
    "task": "binary_classification",
    "label_mapping": {
        "sagittal": {"0": "Anterior", "1": "Non-anterior (Normal+Posterior)"},
        "frontal":  {"0": "Non-lateral (Anterior+Normal)", "1": "Lateral"},
    },
    "data": {
        "total_crops": len(all_crops),
        "unique_patients": len(set(groups_all)),
        "sagittal_distribution": dict(Counter(y_all_sag_bin.tolist())),
        "frontal_distribution":  dict(Counter(y_all_fr_bin.tolist())),
    },
    "feature_extraction": {
        "backbone": "ResNet18 (ImageNet, frozen)",
        "slice_positions": SLICE_POSITIONS,
        "mean_pooled_dim": int(X_all_mean.shape[1]),
        "concat_dim": int(X_all_concat.shape[1]),
        "best_feature_set": best_feat,
    },
    "svm_cv_results": {},
    "ensemble_cv_results": {},
    "comparison": comparison_table,
}

for key, res in cv_results.items():
    v5_analysis["svm_cv_results"][key] = {
        "fold_accs": [round(a, 4) for a in res["fold_accs"]],
        "mean_acc": round(res["mean_acc"], 4),
        "std_acc":  round(res["std_acc"], 4),
        "best_params": [
            {k: (str(v) if not isinstance(v, (int, float, str)) else v)
             for k, v in p.items()}
            for p in res["best_params"]
        ],
    }

for head, res in ensemble_cv.items():
    v5_analysis["ensemble_cv_results"][head] = {
        "fold_accs": [round(a, 4) for a in res["fold_accs"]],
        "mean_acc": round(res["mean_acc"], 4),
        "std_acc":  round(res["std_acc"], 4),
    }

for method, sag_res, fr_res in [
    ("svm_tuned", svm_sag, svm_fr),
    ("ensemble",  ens_sag, ens_fr),
]:
    for head, y_true, res, names in [
        ("sagittal", y_all_sag_bin, sag_res, BINARY_NAMES_SAG),
        ("frontal",  y_all_fr_bin,  fr_res,  BINARY_NAMES_FR),
    ]:
        cm = confusion_matrix(y_true, res["oof_preds"], labels=[0, 1]).tolist()
        report = classification_report(
            y_true, res["oof_preds"],
            target_names=names, labels=[0, 1], zero_division=0, output_dict=True,
        )
        v5_analysis.setdefault("classification_reports", {})[f"{method}_{head}"] = report
        v5_analysis.setdefault("confusion_matrices", {})[f"{method}_{head}"] = cm

v5_analysis["final_model"] = {
    "method": final_method,
    "feature_set": best_feat,
    "pca_components": n_comp,
    "svm_params": {k: str(v) for k, v in bp.items()},
    "pipeline_path": pipeline_path,
}

analysis_path = "training_analysis_v5.json"
with open(analysis_path, "w") as f:
    json.dump(v5_analysis, f, indent=2, ensure_ascii=False, default=str)

print(f"Saved analysis → {analysis_path}")
save_output(analysis_path)

## 5. Approach A: Multi-View Fine-Tuning

Shared ResNet18 backbone (freeze all except layer4) → 3 views → concat 3×512 → FC heads.

Weighted CrossEntropyLoss + label smoothing, аугментации на 2D-срезах.

In [ ]:
class MultiViewDataset(Dataset):
    """Loads .npy crop, extracts 3 central slices, applies 2D augmentations."""

    def __init__(self, records, is_train=True):
        self.records = records
        self.is_train = is_train

        self.augment = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(15),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ]) if is_train else None

    def __len__(self):
        return len(self.records)

    def _to_tensor(self, slice_2d):
        t = torch.from_numpy(slice_2d).float().unsqueeze(0)  # (1, H, W)
        t = F.interpolate(t.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)[0]
        t = t.expand(3, -1, -1).clone()
        if self.is_train and self.augment:
            t = self.augment(t)
        t = (t - IMAGENET_MEAN) / IMAGENET_STD
        return t

    def __getitem__(self, idx):
        rec = self.records[idx]
        views = extract_views(rec["crop_path"])  # (3, 128, 128)

        if self.is_train:
            noise_std = random.uniform(0, 0.02)
            views = views + np.random.normal(0, noise_std, views.shape).astype(np.float32)
            np.clip(views, 0, 1, out=views)

        axial    = self._to_tensor(views[0])
        coronal  = self._to_tensor(views[1])
        sagittal = self._to_tensor(views[2])

        labels = torch.tensor([rec["sag"], rec["fr"]], dtype=torch.long)
        return axial, coronal, sagittal, labels

In [ ]:
class MultiViewTMJ(nn.Module):
    """Shared ResNet18 backbone processes 3 views, concat features, 2 classification heads."""

    def __init__(self, num_classes=3, dropout=0.5):
        super().__init__()
        resnet = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # → (B, 512, 1, 1)

        # Freeze everything except layer4
        for name, param in self.backbone.named_parameters():
            if not name.startswith("7."):  # layer4 = children()[7]
                param.requires_grad = False

        feat_dim = 512 * 3  # 3 views

        def _head():
            return nn.Sequential(
                nn.Linear(feat_dim, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout),
                nn.Linear(128, num_classes),
            )

        self.head_sag = _head()
        self.head_fr  = _head()

    def forward(self, axial, coronal, sagittal):
        f1 = self.backbone(axial).flatten(1)     # (B, 512)
        f2 = self.backbone(coronal).flatten(1)
        f3 = self.backbone(sagittal).flatten(1)
        fused = torch.cat([f1, f2, f3], dim=1)   # (B, 1536)
        return self.head_sag(fused), self.head_fr(fused)

In [ ]:
BATCH_SIZE = 16
EPOCHS = 80
LR = 3e-4
WEIGHT_DECAY = 1e-4
LR_PATIENCE = 8
EARLY_STOPPING = 25
LABEL_SMOOTHING = 0.1

_dl_workers = 0 if IN_COLAB else min(4, max(1, (os.cpu_count() or 8) // 2))
train_ds = MultiViewDataset(train_crops, is_train=True)
val_ds   = MultiViewDataset(val_crops,   is_train=False)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=_dl_workers, pin_memory=torch.cuda.is_available(),
    persistent_workers=_dl_workers > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=_dl_workers, pin_memory=torch.cuda.is_available(),
    persistent_workers=_dl_workers > 0,
)

print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches (batch={BATCH_SIZE}, workers={_dl_workers})")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")

In [ ]:
model = MultiViewTMJ(num_classes=NUM_CLASSES).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.1f}%)")

with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device=device)
    s, f = model(dummy, dummy, dummy)
    print(f"Output shapes: sag={s.shape}, fr={f.shape}")

In [ ]:
def _class_weights(counts, num_classes=3):
    total = sum(counts.values())
    return torch.tensor(
        [total / (num_classes * max(counts.get(c, 0), 1)) for c in range(num_classes)],
        dtype=torch.float32,
    )

sag_w = _class_weights(sag_counts)
fr_w  = _class_weights(fr_counts)
print(f"Sagittal weights: {[round(x, 2) for x in sag_w.tolist()]}")
print(f"Frontal  weights: {[round(x, 2) for x in fr_w.tolist()]}")

criterion_sag = nn.CrossEntropyLoss(weight=sag_w.to(device), label_smoothing=LABEL_SMOOTHING)
criterion_fr  = nn.CrossEntropyLoss(weight=fr_w.to(device),  label_smoothing=LABEL_SMOOTHING)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE,
)

In [ ]:
import gc


def run_epoch(model, loader, criterions, optimizer, is_train, epoch):
    crit_sag, crit_fr = criterions
    model.train() if is_train else model.eval()
    phase = "Train" if is_train else "Val"
    running_loss = 0.0
    all_m = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    pbar = tqdm(loader, desc=f"Epoch {epoch} [{phase}]")

    with ctx:
        for axial, coronal, sagittal, labels in pbar:
            axial, coronal, sagittal = axial.to(device), coronal.to(device), sagittal.to(device)
            labels = labels.to(device)

            if is_train:
                optimizer.zero_grad()

            out_sag, out_fr = model(axial, coronal, sagittal)
            loss = crit_sag(out_sag, labels[:, 0]) + crit_fr(out_fr, labels[:, 1])

            if is_train:
                loss.backward()
                optimizer.step()

            acc_s = (out_sag.argmax(1) == labels[:, 0]).float().mean().item()
            acc_f = (out_fr.argmax(1)  == labels[:, 1]).float().mean().item()
            m = {"acc_sagittal": acc_s, "acc_frontal": acc_f,
                 "mean_accuracy": (acc_s + acc_f) / 2, "loss": loss.item()}
            all_m.append(m)
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{m['mean_accuracy']:.3f}")

    avg = {k: float(np.mean([m[k] for m in all_m])) for k in all_m[0]}
    avg["loss"] = running_loss / len(loader)
    return avg

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir = Path(OUTPUT_DIR) / f"multiview_v4_{timestamp}"
exp_dir.mkdir(parents=True, exist_ok=True)

config_a = {
    "approach": "A (multi-view fine-tune)",
    "backbone": "resnet18 (ImageNet)",
    "views": ["axial", "coronal", "sagittal"],
    "trainable_layers": "layer4 + heads",
    "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE, "label_smoothing": LABEL_SMOOTHING,
    "lr_patience": LR_PATIENCE, "early_stopping": EARLY_STOPPING,
    "train_crops": len(train_crops), "val_crops": len(val_crops),
    "device": str(device),
    "augmentations": ["h_flip", "v_flip", "rotation_15", "translate_5pct", "gaussian_noise"],
}
with open(exp_dir / "config.json", "w") as f_cfg:
    json.dump(config_a, f_cfg, indent=2)

criterions = (criterion_sag, criterion_fr)
best_val_acc = -1.0
best_model_path = exp_dir / "best_model.pth"
epochs_no_improve = 0
history = []

print("=" * 60)
print("STARTING TRAINING — Approach A (Multi-View)")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, criterions, optimizer, True, epoch)
    val_m   = run_epoch(model, val_loader,   criterions, None,      False, epoch)

    scheduler.step(val_m["mean_accuracy"])
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch}/{EPOCHS}  "
          f"Train loss={train_m['loss']:.4f} acc={train_m['mean_accuracy']:.3f}  "
          f"Val loss={val_m['loss']:.4f} acc={val_m['mean_accuracy']:.3f}  "
          f"LR={current_lr:.6f}")

    row = {"epoch": epoch, "lr": current_lr}
    row.update({f"train_{k}": v for k, v in train_m.items()})
    row.update({f"val_{k}": v for k, v in val_m.items()})
    history.append(row)

    with open(exp_dir / "metrics.jsonl", "a") as f_log:
        f_log.write(json.dumps(row) + "\n")

    if val_m["mean_accuracy"] > best_val_acc:
        best_val_acc = val_m["mean_accuracy"]
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_accuracy": best_val_acc,
            "val_metrics": val_m,
        }, best_model_path)
        print(f"  >>> Saved best model (acc={best_val_acc:.3f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{EARLY_STOPPING})")

    if epochs_no_improve >= EARLY_STOPPING:
        print(f"\nEarly stopping at epoch {epoch}")
        break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE — Best val accuracy: {best_val_acc:.3f}")
print(f"{'=' * 60}")

In [ ]:
# Training curves
import pandas as pd

df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(df["epoch"], df["train_loss"], label="Train", lw=2)
axes[0].plot(df["epoch"], df["val_loss"], label="Val", lw=2)
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df["epoch"], df["train_mean_accuracy"], label="Train", lw=2)
axes[1].plot(df["epoch"], df["val_mean_accuracy"], label="Val", lw=2)
axes[1].axhline(best_val_acc, color="r", ls="--", alpha=0.5, label=f"Best={best_val_acc:.3f}")
axes[1].set(title="Mean Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

for name in HEAD_NAMES:
    axes[2].plot(df["epoch"], df[f"val_acc_{name}"], label=name, lw=1.5)
axes[2].set(title="Per-Head Val Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(exp_dir / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices for Approach A (best model)
ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_acc={ckpt['best_val_accuracy']:.3f})")

model.eval()
a_preds_sag, a_preds_fr = [], []
a_true_sag, a_true_fr = [], []

with torch.no_grad():
    for axial, coronal, sagittal, labels in val_loader:
        axial, coronal, sagittal = axial.to(device), coronal.to(device), sagittal.to(device)
        out_sag, out_fr = model(axial, coronal, sagittal)
        a_preds_sag.extend(out_sag.argmax(1).cpu().tolist())
        a_preds_fr.extend(out_fr.argmax(1).cpu().tolist())
        a_true_sag.extend(labels[:, 0].tolist())
        a_true_fr.extend(labels[:, 1].tolist())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, true, pred, title in [
    (axes[0], a_true_sag, a_preds_sag, "Sagittal"),
    (axes[1], a_true_fr,  a_preds_fr,  "Frontal"),
]:
    cm = confusion_matrix(true, pred, labels=CLASS_LABELS)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    acc = accuracy_score(true, pred)
    ax.set(title=f"{title} — Approach A (acc={acc:.3f})", xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.savefig(exp_dir / "confusion_matrices_A.png", dpi=150, bbox_inches="tight")
plt.show()

for head, true, pred in [("SAGITTAL", a_true_sag, a_preds_sag),
                          ("FRONTAL",  a_true_fr,  a_preds_fr)]:
    print(f"\n{head} classification report")
    print("=" * 60)
    print(classification_report(true, pred, labels=CLASS_LABELS,
                                target_names=CLASS_NAMES, zero_division=0))

acc_a_sag = accuracy_score(a_true_sag, a_preds_sag)
acc_a_fr  = accuracy_score(a_true_fr,  a_preds_fr)
acc_a_mean = (acc_a_sag + acc_a_fr) / 2
print(f"Approach A — sag={acc_a_sag:.3f}  fr={acc_a_fr:.3f}  mean={acc_a_mean:.3f}")

## 6. Comparison

v1 (3D baseline), v3 (3D balanced), Approach C (frozen + classical ML), Approach A (multi-view fine-tune).

In [ ]:
comparison = {
    "v1 (3D baseline)": {"sag": 0.875, "fr": 0.594, "mean": 0.734,
                         "note": "majority-class collapse (sagittal)"},
    "v3 (3D balanced)": {"sag": 0.875, "fr": 0.594, "mean": 0.734,
                         "note": "same peak, less overfit"},
}

for clf_name, res in results_c.items():
    comparison[f"C: {clf_name}"] = {
        "sag": res["sagittal"]["accuracy"],
        "fr":  res["frontal"]["accuracy"],
        "mean": res["mean_accuracy"],
        "note": "frozen ResNet18 features",
    }

comparison["A: Multi-View FT"] = {
    "sag": acc_a_sag, "fr": acc_a_fr, "mean": acc_a_mean,
    "note": f"best epoch {ckpt['epoch']}",
}

print(f"{'Method':<25s}  {'Sag':>6s}  {'Fr':>6s}  {'Mean':>6s}  Note")
print("-" * 80)
for name, m in comparison.items():
    print(f"{name:<25s}  {m['sag']:6.3f}  {m['fr']:6.3f}  {m['mean']:6.3f}  {m.get('note', '')}")

print(f"\nBest classical ML: {best_clf_name} (mean={best_c['mean_accuracy']:.3f})")
print(f"Best fine-tuned:   Approach A (mean={acc_a_mean:.3f})")

## 7. Export

In [ ]:
import shutil

models_dest = Path(OUTPUT_DIR) / "trained_models"
models_dest.mkdir(parents=True, exist_ok=True)

if best_model_path.exists():
    dest = models_dest / f"position_2d_multiview_{timestamp}.pth"
    shutil.copy2(best_model_path, dest)
    print(f"Model saved: {dest}")
    save_output(dest)

curves_src = exp_dir / "training_curves.png"
if curves_src.exists():
    shutil.copy2(curves_src, models_dest / f"training_curves_2d_{timestamp}.png")

In [ ]:
best_row = df.loc[df["val_mean_accuracy"].idxmax()] if len(history) > 0 else {}

approach_c_summary = {}
for clf_name, res in results_c.items():
    approach_c_summary[clf_name] = {
        "mean_accuracy": res["mean_accuracy"],
        "sagittal_accuracy": res["sagittal"]["accuracy"],
        "frontal_accuracy": res["frontal"]["accuracy"],
        "sagittal_report": res["sagittal"]["report"],
        "frontal_report": res["frontal"]["report"],
        "sagittal_cm": res["sagittal"]["confusion_matrix"],
        "frontal_cm": res["frontal"]["confusion_matrix"],
    }

analysis_report = {
    "approach_c": {
        "best_classifier": best_clf_name,
        "results": approach_c_summary,
    },
    "approach_a": {
        "config": config_a,
        "total_epochs": len(history),
        "best_epoch": int(best_row.get("epoch", 0)),
        "best_val_accuracy": round(float(best_row.get("val_mean_accuracy", 0)), 4),
        "best_val_acc_sagittal": round(float(best_row.get("val_acc_sagittal", 0)), 4),
        "best_val_acc_frontal": round(float(best_row.get("val_acc_frontal", 0)), 4),
        "history": history,
        "classification_report": {
            "sagittal": classification_report(a_true_sag, a_preds_sag,
                labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0, output_dict=True),
            "frontal": classification_report(a_true_fr, a_preds_fr,
                labels=CLASS_LABELS, target_names=CLASS_NAMES, zero_division=0, output_dict=True),
        },
        "confusion_matrix": {
            "sagittal": confusion_matrix(a_true_sag, a_preds_sag, labels=CLASS_LABELS).tolist(),
            "frontal":  confusion_matrix(a_true_fr,  a_preds_fr,  labels=CLASS_LABELS).tolist(),
        },
    },
    "comparison": comparison,
}

report_path = exp_dir / "training_analysis.json"
with open(report_path, "w") as f_report:
    json.dump(analysis_report, f_report, indent=2, ensure_ascii=False)

save_output(report_path)
print(f"\nSaved: {report_path}")